# 12 · SciPy: estatística, otimização e matrizes esparsas

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados · *notebook complementar*

O SciPy é a biblioteca científica que fica logo acima do NumPy. Você já a usou sem perceber: a matriz que o TF-IDF devolveu na aula 05 é do SciPy, e o algoritmo que ajustou a regressão logística do scikit-learn também. Esta aula abre três das suas gavetas, as que um cientista de dados mais usa.

## O que você será capaz de fazer ao final

1. Trabalhar com distribuições de probabilidade e calcular intervalos de confiança com `scipy.stats`.
2. Aplicar testes de hipótese e distinguir **significância estatística** de **relevância prática**.
3. Usar reamostragem (*bootstrap* e permutação) quando não há fórmula pronta.
4. Minimizar funções com `scipy.optimize`, incluindo o ajuste de um modelo "à mão" e de uma curva.
5. Entender e manipular matrizes esparsas com `scipy.sparse`, sem estourar a memória.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

In [ ]:
import time
import warnings
from pathlib import Path
from urllib.request import urlretrieve

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy import optimize, sparse, stats

SEMENTE = 42
print("scipy", scipy.__version__, "· numpy", np.__version__)

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11})

URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False).dropna(subset=["recommend_to_a_friend"])
df["nao_recomenda"] = (df["recommend_to_a_friend"] == "No").astype(int)
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"].fillna("")
df["palavras"] = df["texto"].str.split().str.len()
df["exclamacoes"] = df["texto"].str.count("!")
print(f"{len(df)} avaliações · {df['nao_recomenda'].mean():.1%} não recomendam")

## 1. O que há no SciPy

| Módulo | Para quê | Nesta aula |
|---|---|---|
| `scipy.stats` | distribuições, testes de hipótese, reamostragem | seções 2 a 5 |
| `scipy.optimize` | minimizar funções, ajustar curvas, resolver equações | seção 6 |
| `scipy.sparse` | matrizes com quase todos os elementos iguais a zero | seção 7 |
| `scipy.spatial` | distâncias, vizinhos mais próximos | — |
| `scipy.cluster` | agrupamento hierárquico | — |
| `scipy.signal`, `scipy.fft` | sinais, filtros, transformada de Fourier | — |
| `scipy.interpolate`, `scipy.integrate` | interpolação, integração numérica | — |
| `scipy.linalg` | álgebra linear além do NumPy | — |

A relação com o resto do ecossistema: o **NumPy** dá o array; o **SciPy** dá os algoritmos numéricos clássicos sobre arrays; o **scikit-learn** usa os dois para construir modelos; o **statsmodels** cobre a modelagem estatística com tabelas de resultados no estilo dos pacotes de estatística.

## 2. Distribuições de probabilidade

Em `scipy.stats`, cada distribuição é um objeto com os mesmos métodos:

| Método | Devolve | Pergunta que responde |
|---|---|---|
| `pmf(k)` ou `pdf(x)` | probabilidade (ou densidade) no ponto | "qual a chance de exatamente 60?" |
| `cdf(x)` | probabilidade acumulada até `x` | "qual a chance de 60 ou menos?" |
| `sf(x)` | 1 − `cdf(x)` | "qual a chance de mais de 60?" |
| `ppf(q)` | o valor que deixa a proporção `q` abaixo (inversa da `cdf`) | "abaixo de que valor ficam 95% dos casos?" |
| `rvs(size)` | amostras aleatórias | simulação |

Exemplo: 27,2% das avaliações são de quem não recomenda. Em um lote de 200 avaliações sorteadas, quantas dessas esperamos ver? O número segue uma distribuição **binomial**.

In [ ]:
lote = stats.binom(n=200, p=0.272)                    # a distribuição fica "congelada" com seus parâmetros

print("média:", round(lote.mean(), 1), "· desvio padrão:", round(lote.std(), 2))
print("P(exatamente 54)  =", round(lote.pmf(54), 4))
print("P(60 ou menos)    =", round(lote.cdf(60), 4))
print("P(mais de 60)     =", round(lote.sf(60), 4))
print("95% dos lotes têm até", int(lote.ppf(0.95)), "avaliações negativas")
print("cinco lotes simulados:", lote.rvs(5, random_state=SEMENTE))

In [ ]:
k = np.arange(30, 81)
aproximacao = stats.norm(loc=lote.mean(), scale=lote.std())           # a normal com a mesma média e o mesmo desvio

fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(k, lote.pmf(k), color=AZUL, width=0.8, label="binomial(200; 0,272)")
ax.plot(k, aproximacao.pdf(k), color=LARANJA, lw=2, label="aproximação normal")
ax.set_xlabel("avaliações de quem não recomenda, em um lote de 200"); ax.set_ylabel("probabilidade"); ax.legend(frameon=False)
ax.set_title("Com 200 observações, a binomial já tem forma de sino")
print("O famoso 1,96:", round(stats.norm.ppf(0.975), 4), "(o valor que deixa 2,5% da normal padrão acima dele)")

A aproximação pela normal é a origem da fórmula que usamos na aula 03 para a margem de erro de uma proporção: `1,96 × √(p(1−p)/n)`.

### ✏️ Exercício 1 (★): perguntas a uma distribuição

Para a mesma distribuição (`n=200`, `p=0.272`), calcule:

- `p_ex1`: a probabilidade de um lote ter **70 ou mais** avaliações de quem não recomenda;
- `limite_ex1`: o menor número `k` tal que pelo menos 99% dos lotes têm `k` ou menos.

In [ ]:
# ✏️ Exercício 1
# SEU CÓDIGO AQUI
p_ex1 = ...
limite_ex1 = ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
_pmf = stats.binom.pmf(np.arange(201), 200, 0.272)
assert abs(p_ex1 - _pmf[70:].sum()) < 1e-9, "atenção ao limite: '70 ou mais' inclui o 70"
assert _pmf[: limite_ex1 + 1].sum() >= 0.99 > _pmf[:limite_ex1].sum(), "limite_ex1 deve ser o menor k com P(X <= k) >= 0,99"
print(f"Exercício 1 correto: P(70 ou mais) = {p_ex1:.4f} · 99% dos lotes têm até {limite_ex1}.")

## 3. Intervalos de confiança

Uma taxa medida em uma amostra é uma estimativa. O **intervalo de confiança** diz a faixa de valores compatíveis com os dados. Para proporções há mais de uma fórmula:

- **Wald**: a da aula 03, `p ± 1,96 × √(p(1−p)/n)`. Simples, e ruim para amostras pequenas ou taxas perto de 0 ou de 1.
- **Wilson**: mais confiável nesses casos; é a recomendada para o dia a dia.
- **Exata** (Clopper-Pearson): usa a binomial diretamente; é conservadora (intervalos um pouco largos).

O SciPy calcula as duas últimas a partir de `stats.binomtest`. A de Wald fica para você, no exercício 2.

In [ ]:
por_estado = df.groupby("reviewer_state")["nao_recomenda"].agg(n="size", k="sum")

for estado in ["AC", "PR"]:
    n, k = int(por_estado.loc[estado, "n"]), int(por_estado.loc[estado, "k"])
    resultado = stats.binomtest(k, n)
    wilson, exata = resultado.proportion_ci(method="wilson"), resultado.proportion_ci(method="exact")
    print(f"{estado}: {k} de {n} = {k / n:.1%} · Wilson [{wilson.low:.1%}; {wilson.high:.1%}] · exata [{exata.low:.1%}; {exata.high:.1%}]")

No Paraná, com milhares de avaliações, os dois métodos coincidem e o intervalo é estreito. No Acre, com 87, o intervalo tem cerca de 16 pontos percentuais de largura e os métodos já discordam nas bordas: a taxa do Acre parece a mais baixa do país, e os dados não permitem afirmar isso.

### ✏️ Exercício 2 (★★): o intervalo de Wald, à mão

Escreva `ic_wald(k, n, confianca=0.95)`, que devolve a tupla `(inferior, superior)` do intervalo de Wald para a proporção `k/n`. Use `stats.norm.ppf` para obter o multiplicador (para 95%, é o 1,96).

In [ ]:
# ✏️ Exercício 2
def ic_wald(k, n, confianca=0.95):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
for (_k, _n, _c), _esperado in [((16, 87, 0.95), (0.102503, 0.265313)), ((1692, 6745, 0.95), (0.240507, 0.261199)),
                                ((30, 100, 0.99), (0.181961, 0.418039)), ((30, 100, 0.80), (0.241272, 0.358728))]:
    assert np.allclose(ic_wald(_k, _n, _c), _esperado, atol=1e-5), f"resultado inesperado para k={_k}, n={_n}, confiança={_c}"
_wilson = stats.binomtest(16, 87).proportion_ci(method="wilson")
print("Exercício 2 correto, inclusive para 80% e 99% de confiança.")
print(f"Acre (16 de 87): Wald [{ic_wald(16, 87)[0]:.1%}; {ic_wald(16, 87)[1]:.1%}] · Wilson [{_wilson.low:.1%}; {_wilson.high:.1%}]")

## 4. Testes de hipótese: significância e relevância

Um teste de hipótese responde a uma pergunta estreita: **se não houvesse diferença nenhuma, com que frequência veríamos uma diferença tão grande quanto a observada, só por acaso?** Essa frequência é o **p-valor**. Quando é pequena (por convenção, abaixo de 0,05), diz-se que a diferença é *estatisticamente significante*.

Pergunta do dia: clientes mulheres e homens avaliam diferente?

In [ ]:
com_genero = df.dropna(subset=["reviewer_gender"])
resumo = com_genero.groupby("reviewer_gender").agg(avaliacoes=("overall_rating", "size"), nota_media=("overall_rating", "mean"), nao_recomenda=("nao_recomenda", "mean"))
print(resumo.round(3), "\n")

notas_f = com_genero.loc[com_genero["reviewer_gender"] == "F", "overall_rating"]
notas_m = com_genero.loc[com_genero["reviewer_gender"] == "M", "overall_rating"]

teste_t = stats.ttest_ind(notas_f, notas_m, equal_var=False)                                   # médias (teste t de Welch)
teste_u = stats.mannwhitneyu(notas_f, notas_m)                                                 # sem supor distribuição normal
teste_qui = stats.chi2_contingency(pd.crosstab(com_genero["reviewer_gender"], com_genero["nao_recomenda"]))      # proporções

print(f"nota média:      teste t           p = {teste_t.pvalue:.1e}")
print(f"nota (postos):   Mann-Whitney      p = {teste_u.pvalue:.1e}")
print(f"não recomenda:   qui-quadrado      p = {teste_qui.pvalue:.1e}")

Os três p-valores são minúsculos: a diferença é "altamente significante". Agora olhe o **tamanho** dela.

In [ ]:
diferenca = notas_f.mean() - notas_m.mean()
d_de_cohen = diferenca / com_genero["overall_rating"].std()           # diferença em unidades de desvio padrão
print(f"diferença de nota média: {diferenca:.3f} ponto (em uma escala de 1 a 5) · d de Cohen: {d_de_cohen:.3f}")
print(f"diferença na taxa de 'não recomenda': {(resumo.loc['M', 'nao_recomenda'] - resumo.loc['F', 'nao_recomenda']) * 100:.1f} ponto percentual")

Seis centésimos de ponto em uma escala de cinco. Um *d* de Cohen de 0,04, quando 0,2 já é considerado um efeito pequeno. Nenhuma decisão de negócio mudaria por causa dessa diferença.

O p-valor ficou minúsculo porque a amostra é enorme. **O p-valor mede quanta evidência há de que a diferença não é zero; ele não mede se a diferença importa.** Veja o mesmo teste com amostras menores:

In [ ]:
linhas = []
for n in [200, 2000, 20000, len(notas_f)]:
    a, b = notas_f.sample(n, random_state=0), notas_m.sample(n, random_state=0)
    linhas.append({"avaliações por grupo": n, "diferença de nota": a.mean() - b.mean(), "p-valor": stats.ttest_ind(a, b, equal_var=False).pvalue})
pd.DataFrame(linhas).round(4)

A diferença verdadeira é a mesma nas quatro linhas. O que muda é a capacidade do teste de enxergá-la: com 200 avaliações por grupo ela é invisível; com dezenas de milhares, é "significante". Em ciência de dados, com bases grandes, a regra prática é:

> 📌 **Relate sempre o tamanho do efeito e o seu intervalo de confiança.** O p-valor, sozinho, diz pouco.

| Qual teste? | Função |
|---|---|
| Duas médias | `stats.ttest_ind` (com `equal_var=False`) |
| Duas distribuições, sem supor normalidade; dados ordinais | `stats.mannwhitneyu` |
| Antes e depois, nos mesmos indivíduos | `stats.ttest_rel`, `stats.wilcoxon` |
| Proporções; associação entre duas variáveis categóricas | `stats.chi2_contingency` |
| Três ou mais grupos | `stats.f_oneway`, `stats.kruskal` |
| Associação entre duas variáveis numéricas | `stats.pearsonr`, `stats.spearmanr` |

### 🔮 Preveja: procurando até achar

Vamos dividir as avaliações em dois grupos **por sorteio**, 200 vezes, e testar a cada vez se a nota média difere. Os grupos não têm nenhuma diferença real. Em quantas das 200 divisões o teste dirá "significante"?

In [ ]:
rng = np.random.default_rng(0)
notas = df["overall_rating"].to_numpy()
p_valores = []
for _ in range(200):
    sorteio = rng.integers(0, 2, len(notas)).astype(bool)
    p_valores.append(stats.ttest_ind(notas[sorteio], notas[~sorteio]).pvalue)
p_valores = np.array(p_valores)

print(f"'Significantes' a 5%: {(p_valores < 0.05).sum()} de 200 ({(p_valores < 0.05).mean():.1%})")
print(f"Com a correção de Bonferroni (limite de 0,05 / 200): {(p_valores < 0.05 / 200).sum()}")

Cerca de 5%, como a definição prevê: o limite de 0,05 **é** a taxa de alarmes falsos que aceitamos em cada teste. A correção de Bonferroni derruba os alarmes de 11 para 1: reduz muito, e não zera. Quem testa vinte hipóteses espera um "achado" por puro acaso. Comparar todas as categorias, todos os estados e todos os meses, e relatar só o que deu significante, é a forma mais comum de produzir resultados que não se repetem.

As defesas: decidir a pergunta **antes** de olhar os dados; corrigir o limite quando há vários testes (Bonferroni: dividir 0,05 pelo número de testes; ou `stats.false_discovery_control`); e, de novo, olhar o tamanho do efeito.

### ✏️ Exercício 3 (★★): duas categorias

Escreva `comparar_taxas(categoria_a, categoria_b)`, que compara a taxa de "não recomenda" de duas categorias de `site_category_lv1` e devolve a tupla `(diferenca, p_valor)`: a taxa da categoria A menos a da B, e o p-valor do teste qui-quadrado sobre a tabela 2 × 2.

In [ ]:
# ✏️ Exercício 3
def comparar_taxas(categoria_a, categoria_b):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_d1, _p1 = comparar_taxas("Móveis", "Livros")
assert 0.30 < _d1 < 0.33 and _p1 < 1e-50, "Móveis e Livros têm taxas muito diferentes"
_d2, _p2 = comparar_taxas("Livros", "Móveis")
assert abs(_d1 + _d2) < 1e-12 and np.isclose(_p1, _p2), "trocar a ordem inverte o sinal da diferença e mantém o p-valor"
_d3, _p3 = comparar_taxas("Beleza e Perfumaria", "Utilidades Domésticas")
assert abs(_d3) < 0.01 and _p3 > 0.05
print(f"Exercício 3 correto.\n  Móveis − Livros:                             {_d1:+.3f}  (p = {_p1:.0e})\n  Beleza e Perfumaria − Utilidades Domésticas: {_d3:+.3f}  (p = {_p3:.2f})")

**Pergunta escrita.** A diferença entre Beleza e Utilidades Domésticas "não é significante". Isso prova que as duas categorias têm a mesma taxa? E a diferença entre Móveis e Livros: o que, além do p-valor, você informaria a um gestor?

✏️ **Sua resposta:**

_Escreva aqui._

## 5. Reamostragem: quando não há fórmula

Para a média existe uma fórmula de erro padrão. Para a mediana, para a diferença entre dois percentis ou para a AUC de um modelo, não há fórmula simples. O ***bootstrap*** resolve com força bruta: sorteia, **com reposição**, milhares de amostras do mesmo tamanho a partir dos próprios dados, recalcula a estatística em cada uma e observa quanto ela varia.

In [ ]:
palavras_negativas = df.loc[df["nao_recomenda"] == 1, "palavras"].sample(500, random_state=SEMENTE).to_numpy()

reamostragem = stats.bootstrap((palavras_negativas,), np.median, n_resamples=5000, method="percentile", random_state=SEMENTE)
ic = reamostragem.confidence_interval
print(f"Mediana de palavras em 500 avaliações negativas: {np.median(palavras_negativas):.0f} · IC de 95% por bootstrap: [{ic.low:.0f}; {ic.high:.0f}]")
print(f"(mediana em todas as avaliações negativas da base: {df.loc[df['nao_recomenda'] == 1, 'palavras'].median():.0f})")

O **teste de permutação** segue a mesma lógica para comparar grupos: embaralha os rótulos dos grupos milhares de vezes e conta em quantas a diferença embaralhada é tão grande quanto a real. É o experimento da seção anterior, com sorteios, transformado em teste.

In [ ]:
palavras_positivas = df.loc[df["nao_recomenda"] == 0, "palavras"].sample(500, random_state=SEMENTE).to_numpy()


def diferenca_de_medianas(a, b):
    return np.median(a) - np.median(b)


permutacao = stats.permutation_test((palavras_negativas, palavras_positivas), diferenca_de_medianas, n_resamples=5000, random_state=SEMENTE)
print(f"Diferença de medianas: {permutacao.statistic:.0f} palavras · p-valor por permutação: {permutacao.pvalue:.4f}")
print(f"Correlação entre tamanho do texto e nota: Pearson {stats.pearsonr(df['palavras'], df['overall_rating']).statistic:.2f} · Spearman {stats.spearmanr(df['palavras'], df['overall_rating']).statistic:.2f}")

Quem não recomenda escreve mais (o padrão que apareceu nas aulas 04 e 06), e aqui a diferença é grande também em termos práticos: a mediana é cerca de 50% maior.

### ✏️ Exercício 4 (★★): a nota média de um estado pequeno

O Acre tem 87 avaliações. Calcule `ic_ex4`, o intervalo de confiança de 95% para a **nota média** do Acre, por *bootstrap* (`method="percentile"`, 5000 reamostragens, `random_state=SEMENTE`). Guarde o objeto `confidence_interval` devolvido pelo SciPy.

In [ ]:
notas_ac = df.loc[df["reviewer_state"] == "AC", "overall_rating"].to_numpy()

In [ ]:
# ✏️ Exercício 4
# SEU CÓDIGO AQUI
ic_ex4 = ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
assert ic_ex4.low < notas_ac.mean() < ic_ex4.high, "o intervalo deve conter a média da amostra"
_classico = stats.t.interval(0.95, df=len(notas_ac) - 1, loc=notas_ac.mean(), scale=stats.sem(notas_ac))
assert abs(ic_ex4.low - _classico[0]) < 0.08 and abs(ic_ex4.high - _classico[1]) < 0.08, "o intervalo está longe do obtido pela fórmula clássica: a estatística é a média?"
print(f"Exercício 4 correto: nota média do Acre {notas_ac.mean():.2f} · bootstrap [{ic_ex4.low:.2f}; {ic_ex4.high:.2f}] · fórmula clássica (t) [{_classico[0]:.2f}; {_classico[1]:.2f}]")
print(f"Nota média do país: {df['overall_rating'].mean():.2f}")

## 6. Otimização

Treinar um modelo é **minimizar uma função** (a perda) em relação aos parâmetros. Na aula 07 você fez isso com a descida do gradiente. O `scipy.optimize` reúne os algoritmos clássicos, que o scikit-learn usa por baixo.

### 6.1 Uma variável: o limiar de menor custo

Na aula 05 escolhemos o limiar de decisão percorrendo uma grade de valores. É um problema de minimização em uma variável. Suponha que deixar passar um cliente insatisfeito custe 5, e acionar o atendimento sem necessidade custe 1.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

treino, valid = train_test_split(df, test_size=0.2, stratify=df["nao_recomenda"], random_state=SEMENTE)
modelo_texto = make_pipeline(TfidfVectorizer(min_df=5), LogisticRegression(max_iter=1000)).fit(treino["texto"], treino["nao_recomenda"])
prob_valid, y_valid = modelo_texto.predict_proba(valid["texto"])[:, 1], valid["nao_recomenda"].to_numpy()


def custo(limiar):
    """Custo médio por avaliação: 5 por insatisfeito não detectado, 1 por alarme falso."""
    alarme = prob_valid >= limiar
    return (5 * (~alarme & (y_valid == 1)).sum() + 1 * (alarme & (y_valid == 0)).sum()) / len(y_valid)


grade = np.linspace(0.01, 0.99, 99)
custos = [custo(limiar) for limiar in grade]
otimo = optimize.minimize_scalar(custo, bounds=(0.01, 0.99), method="bounded")

fig, ax = plt.subplots(figsize=(6.5, 3.1))
ax.plot(grade, custos, color=AZUL)
ax.axvline(otimo.x, color=LARANJA, ls="--")
ax.set_xlabel("limiar de decisão"); ax.set_ylabel("custo médio por avaliação")
ax.set_title(f"Custo mínimo com limiar perto de {otimo.x:.2f}")
print(f"grade de 99 valores:  limiar {grade[int(np.argmin(custos))]:.2f} · custo {min(custos):.4f}")
print(f"minimize_scalar:      limiar {otimo.x:.4f} · custo {otimo.fun:.4f} · {otimo.nfev} avaliações da função")
print(f"limiar padrão de 0,5: custo {custo(0.5):.4f}")

O otimizador chegou ao mesmo lugar com um quinto das avaliações da função. Nesta função barata, tanto faz; quando cada avaliação é um treino de modelo, a economia decide o que é viável.

> ⚠️ Esta função de custo é feita de degraus (muda só quando o limiar cruza a probabilidade de alguma avaliação). Otimizadores supõem funções suaves; aqui funcionou porque há milhares de degraus pequenos. Em funções com poucos degraus ou vários mínimos, a grade é mais segura.

### 6.2 Várias variáveis: uma regressão logística "à mão"

Agora o problema de verdade: encontrar os pesos que minimizam a perda logarítmica. Usamos duas colunas numéricas, padronizadas, mais o termo constante.

Para `optimize.minimize`, escrevemos duas funções: a **perda** e o seu **gradiente**. (Para a regressão logística o gradiente tem fórmula conhecida: `Xᵀ(p − y) / n`.)

In [ ]:
colunas = treino[["palavras", "exclamacoes"]].to_numpy(dtype=float)
Z = np.column_stack([np.ones(len(colunas)), (colunas - colunas.mean(axis=0)) / colunas.std(axis=0)])      # constante + 2 colunas padronizadas
alvo = treino["nao_recomenda"].to_numpy()


def perda(w):
    z = Z @ w
    return np.mean(np.logaddexp(0, z) - alvo * z)              # perda logarítmica, escrita de forma numericamente estável


def gradiente(w):
    p = 1 / (1 + np.exp(-(Z @ w)))
    return Z.T @ (p - alvo) / len(alvo)


inicio = time.perf_counter()
resultado = optimize.minimize(perda, x0=np.zeros(3), jac=gradiente, method="L-BFGS-B")
print(f"L-BFGS com gradiente: pesos {resultado.x.round(4)} · perda {resultado.fun:.5f} · {resultado.nit} iterações · {(time.perf_counter() - inicio) * 1000:.0f} ms")

sem_gradiente = optimize.minimize(perda, x0=np.zeros(3), method="Nelder-Mead")
print(f"Nelder-Mead, sem gradiente: pesos {sem_gradiente.x.round(4)} · {sem_gradiente.nfev} avaliações da perda")

do_sklearn = LogisticRegression(C=np.inf, max_iter=1000).fit(Z[:, 1:], alvo)           # C infinito: sem regularização
print(f"scikit-learn:         pesos {np.r_[do_sklearn.intercept_, do_sklearn.coef_[0]].round(4)}")

Três leituras:

- Os pesos encontrados com o SciPy coincidem com os do scikit-learn (até a terceira casa decimal, que é a tolerância padrão dele). **Não há mágica dentro do `fit`**: há uma perda, um gradiente e um otimizador. O método padrão da `LogisticRegression` é justamente o L-BFGS.
- Com o gradiente, bastaram poucas iterações. O L-BFGS usa o histórico dos gradientes para estimar a curvatura da função, e por isso anda muito mais depressa que a descida do gradiente simples da aula 07.
- Sem o gradiente (Nelder-Mead), o resultado é o mesmo, com dezenas de vezes mais avaliações da perda. Com milhões de parâmetros, seria inviável: é por isso que o *autograd* importa tanto.

### ✏️ Exercício 5 (★★★): acrescentar regularização

Na aula 05 a regressão logística tinha o hiperparâmetro `C`. Ele controla uma **penalidade** somada à perda, que puxa os pesos para zero. Escreva `perda_l2(w, lam)` e `gradiente_l2(w, lam)`:

- a perda é a anterior **mais** `lam` vezes a soma dos quadrados dos pesos, **sem** incluir o termo constante `w[0]`;
- o gradiente é o anterior mais `2 · lam · w` nas posições 1 em diante.

Depois, minimize com `lam = 0.01` e guarde o vetor de pesos em `pesos_ex5`. Passe `lam` ao otimizador com `args=(0.01,)` e use `options={"gtol": 1e-10, "ftol": 1e-14}`, para um resultado bem preciso.

In [ ]:
# ✏️ Exercício 5
def perda_l2(w, lam):
    # SEU CÓDIGO AQUI
    ...


def gradiente_l2(w, lam):
    # SEU CÓDIGO AQUI
    ...


# SEU CÓDIGO AQUI
pesos_ex5 = ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
_w = np.array([0.3, -0.2, 0.5])
assert np.isclose(perda_l2(_w, 0.0), perda(_w)) and np.isclose(perda_l2(_w, 2.0) - perda(_w), 2.0 * (0.04 + 0.25)), "a penalidade não deve incluir o termo constante"
_numerico = optimize.approx_fprime(_w, perda_l2, 1e-7, 0.5)                    # gradiente por diferenças finitas, para conferir o seu
assert np.allclose(gradiente_l2(_w.copy(), 0.5), _numerico, atol=1e-5), "o gradiente não corresponde à perda"
_C = 1 / (2 * 0.01 * len(alvo))                                                # a penalidade equivalente, na convenção do scikit-learn
_sk = LogisticRegression(C=_C, max_iter=5000, tol=1e-10).fit(Z[:, 1:], alvo)
assert np.allclose(pesos_ex5, np.r_[_sk.intercept_, _sk.coef_[0]], atol=1e-4), "os pesos diferem dos do scikit-learn com a penalidade equivalente"
print(f"Exercício 5 correto: pesos {pesos_ex5.round(4)} = scikit-learn com C = {_C:.6f}")
print(f"Sem penalidade, os pesos eram {resultado.x.round(4)}: a regularização encolheu os dois coeficientes.")

### 6.3 Ajuste de curvas: quanto dado é suficiente?

`optimize.curve_fit` ajusta os parâmetros de uma função qualquer a pontos medidos. Um uso prático em aprendizagem de máquina: medir o desempenho do modelo com quantidades crescentes de dados, ajustar uma curva e estimar quanto se ganharia com mais dados, **antes** de pagar pela rotulagem.

In [ ]:
embaralhado = treino.sample(frac=1, random_state=0)
tamanhos = [100, 300, 1000, 3000, 10000, 30000]


def auc_com(n):
    """AUC de validação de um TF-IDF + regressão logística treinado com as n primeiras avaliações."""
    m = make_pipeline(TfidfVectorizer(min_df=1 if n < 1000 else 2), LogisticRegression(max_iter=1000))
    m.fit(embaralhado["texto"].iloc[:n], embaralhado["nao_recomenda"].iloc[:n])
    return roc_auc_score(y_valid, m.predict_proba(valid["texto"])[:, 1])


aucs = [auc_com(n) for n in tamanhos]


def curva(n, teto, b, c):
    """Lei de potência: o desempenho se aproxima de um teto à medida que n cresce."""
    return teto - b * n ** (-c)


parametros, covariancia = optimize.curve_fit(curva, tamanhos, aucs, p0=[0.97, 0.5, 0.5], maxfev=20000)
incertezas = np.sqrt(np.diag(covariancia))
print("AUC medida:", dict(zip(tamanhos, np.round(aucs, 4))))
print(f"teto estimado: {parametros[0]:.4f} ± {incertezas[0]:.4f}")

In [ ]:
n_total = len(embaralhado)
previsto, real = curva(n_total, *parametros), auc_com(n_total)          # o teste da extrapolação: treinar com tudo

eixo = np.logspace(2, np.log10(n_total), 100)
fig, ax = plt.subplots(figsize=(6.8, 3.3))
ax.plot(eixo, curva(eixo, *parametros), color=AZUL, label="curva ajustada")
ax.scatter(tamanhos, aucs, color=AZUL, zorder=3, label="medido (usado no ajuste)")
ax.scatter([n_total], [real], color=LARANJA, zorder=3, label="medido com todos os dados")
ax.set_xscale("log"); ax.set_xlabel("avaliações de treino (escala logarítmica)"); ax.set_ylabel("AUC na validação"); ax.legend(frameon=False, fontsize=8)
ax.set_title("A curva acerta a tendência e erra o teto por pouco")
print(f"Com {n_total} avaliações: previsto pela curva {previsto:.4f} · medido {real:.4f}")

A curva ajustada com até 30 mil exemplos previu bem a ordem de grandeza do que se obteria com 100 mil: o ganho de triplicar os dados é de milésimos. Como decisão ("vale rotular mais?"), ela acertou.

Como número, ela errou para baixo: o valor medido com todos os dados ficou **acima do teto** que a curva estimou. Extrapolar é sempre arriscado: a forma escolhida para a curva é uma hipótese, e a incerteza informada pelo ajuste só vale se a hipótese estiver certa.

## 7. Matrizes esparsas

O TF-IDF transforma cada avaliação em um vetor com uma posição por palavra do vocabulário. Quase todas as posições valem zero, porque cada avaliação usa poucas dezenas de palavras. Guardar os zeros seria um desperdício absurdo.

In [ ]:
vetorizador, classificador = modelo_texto[0], modelo_texto[-1]          # as duas etapas do pipeline
X = vetorizador.transform(df["texto"])

memoria_esparsa = (X.data.nbytes + X.indices.nbytes + X.indptr.nbytes) / 1e6
memoria_densa = X.shape[0] * X.shape[1] * 8 / 1e9
print(type(X).__name__, X.shape, X.dtype)
print(f"elementos diferentes de zero: {X.nnz} de {X.shape[0] * X.shape[1]} ({X.nnz / (X.shape[0] * X.shape[1]):.2%})")
print(f"memória: {memoria_esparsa:.0f} MB no formato esparso · {memoria_densa:.1f} GB se fosse uma matriz comum")

A mesma informação, em mais de quatrocentas vezes menos memória. Uma matriz comum desse tamanho não caberia no Colab; `X.toarray()` derrubaria a sessão.

### Como funciona: o formato CSR

O formato mais usado, **CSR** (*compressed sparse row*), guarda três vetores:

In [ ]:
pequena = sparse.csr_matrix(np.array([[0, 0, 3, 0],
                                      [1, 0, 0, 2],
                                      [0, 0, 0, 0],
                                      [0, 4, 0, 0]]))
print("data   :", pequena.data, "   ← os valores diferentes de zero, linha por linha")
print("indices:", pequena.indices, "   ← a coluna de cada valor")
print("indptr :", pequena.indptr, " ← onde começa cada linha dentro de data")

A linha `i` ocupa as posições `indptr[i]` até `indptr[i+1]`. A linha 2, vazia, começa e termina na mesma posição.

| Formato | Bom para | Ruim para |
|---|---|---|
| **CSR** | fatiar linhas, produto matriz × vetor; é o que os modelos esperam | fatiar colunas, alterar elementos |
| **CSC** | fatiar colunas | fatiar linhas |
| **COO** | construir a matriz a partir de trios (linha, coluna, valor) | contas |

### Operações que preservam a esparsidade

In [ ]:
linha = X[0]                                                     # uma avaliação: fatiar linhas em CSR é barato
palavras_do_vocabulario = vetorizador.get_feature_names_out()
print("primeira avaliação:", df["texto"].iloc[0][:70])
print("palavras e pesos:  ", {palavras_do_vocabulario[j]: round(v, 2) for j, v in zip(linha.indices, linha.data)})

soma_por_palavra = np.asarray(X.sum(axis=0)).ravel()             # soma por coluna: devolve uma matriz 1 × colunas, que achatamos
print("\npalavras de maior peso total:", list(palavras_do_vocabulario[np.argsort(-soma_por_palavra)[:8]]))

coeficientes = classificador.coef_[0]
log_chances = X[:5] @ coeficientes + classificador.intercept_[0]            # produto esparso × denso: a previsão do modelo, à mão
print("\nlog-chances das 5 primeiras, à mão:    ", log_chances.round(3))
print("as mesmas, pelo modelo:                 ", classificador.decision_function(X[:5]).round(3))

O produto de uma matriz esparsa por um vetor só percorre os elementos diferentes de zero. É por isso que um modelo linear treina em segundos com quase 13 mil colunas.

In [ ]:
# Semelhança entre avaliações: as linhas do TF-IDF já têm norma 1, então o produto interno é o cosseno
consulta = 0
semelhancas = (X @ X[consulta].T).toarray().ravel()              # (132 mil × 13 mil) @ (13 mil × 1): o resultado é pequeno e pode ser denso
mais_parecidas = np.argsort(-semelhancas)[1:4]
print("avaliação:", df["texto"].iloc[consulta][:90])
for i in mais_parecidas:
    print(f"  {semelhancas[i]:.2f}  {df['texto'].iloc[i][:90]}")

Esta é a busca **por palavras em comum**. Compare com a busca semântica da aula 09, que encontra textos de mesmo sentido com palavras diferentes.

> ⚠️ **Armadilhas.** `X.toarray()` em uma matriz grande estoura a memória. Somar um número a uma matriz esparsa (`X + 1`) a tornaria densa, e o SciPy recusa. Padronizar subtraindo a média também destrói a esparsidade: por isso se usa `StandardScaler(with_mean=False)` ou `MaxAbsScaler` com dados esparsos. Atribuir elemento a elemento (`X[i, j] = v`) em CSR é lento.

### ✏️ Exercício 6 (★★): em quantas avaliações cada palavra aparece?

Sem converter `X` em matriz densa, calcule `frequencia_ex6`: uma `Series` indexada pelas palavras do vocabulário com o **número de avaliações** em que cada palavra aparece, ordenada da mais para a menos frequente. Guarde também em `mb_ex6` a memória ocupada por `X`, em megabytes (some os três vetores do formato CSR).

Dica: o vetor `X.indices` tem a coluna de cada elemento diferente de zero; `np.bincount` conta ocorrências.

In [ ]:
# ✏️ Exercício 6
# SEU CÓDIGO AQUI
frequencia_ex6 = ...
mb_ex6 = ...

In [ ]:
# ✅ Verificação do exercício 6 (não altere)
assert len(frequencia_ex6) == X.shape[1] and frequencia_ex6.is_monotonic_decreasing and frequencia_ex6.sum() == X.nnz, "uma contagem por palavra, em ordem decrescente; a soma é o número de elementos diferentes de zero"
_csc = X.tocsc()
for _palavra in list(frequencia_ex6.index[:3]) + list(frequencia_ex6.index[-2:]):
    assert frequencia_ex6[_palavra] == _csc[:, vetorizador.vocabulary_[_palavra]].nnz, f"contagem errada para '{_palavra}'"
assert 25 < mb_ex6 < 40
print(f"Exercício 6 correto: {mb_ex6:.1f} MB. Palavras presentes em mais avaliações:")
print((frequencia_ex6.head(8) / X.shape[0]).map("{:.1%}".format).to_string())

## Para levar

### Erros comuns

| Erro | Consequência | Como evitar |
|---|---|---|
| Relatar só o p-valor | com base grande, tudo "é significante" | tamanho do efeito e intervalo de confiança |
| Ler "não significante" como "igual" | conclusão sem base | olhar o intervalo de confiança da diferença |
| Testar muitas hipóteses e relatar as que deram certo | achados que não se repetem | decidir a pergunta antes; corrigir para múltiplos testes |
| Intervalo de Wald com amostra pequena | intervalo estreito demais, ou fora de 0 a 1 | Wilson |
| Supor normalidade em notas de 1 a 5 ou em contagens | teste inadequado | testes de postos; reamostragem |
| Otimizar sem fornecer o gradiente | centenas de vezes mais avaliações | passar `jac=`; conferir com `approx_fprime` |
| Confiar no ponto inicial em função com vários mínimos | mínimo local | várias partidas; inspecionar a função |
| Extrapolar uma curva ajustada como se fosse medida | previsão com confiança indevida | tratar a forma da curva como hipótese; validar com um ponto a mais |
| `toarray()` em matriz esparsa grande | falta de memória | operar no formato esparso |
| Subtrair a média de dados esparsos | a matriz vira densa | `with_mean=False`, `MaxAbsScaler` |

### Resumo

- `scipy.stats`: distribuições com interface única (`pmf`/`pdf`, `cdf`, `sf`, `ppf`, `rvs`), intervalos de confiança e testes.
- O **p-valor** mede evidência contra "diferença zero". Com muitos dados, relate o **tamanho do efeito**.
- Vários testes multiplicam os alarmes falsos: 5% deles dão "significante" por acaso.
- ***Bootstrap*** e **permutação** substituem fórmulas por reamostragem.
- `scipy.optimize`: `minimize_scalar`, `minimize` (com gradiente, sempre que possível) e `curve_fit`. É o que há dentro do `fit` dos modelos clássicos.
- `scipy.sparse`: guarda só os elementos diferentes de zero. CSR para linhas e produtos; nunca converter para densa sem fazer a conta da memória.

### Para casa

A lista de exercícios está em `12_casa.ipynb`.